# အခန်း ၁၈ (ဆက်လက်): ကျေနပ်သူ၏လက်မှတ်ကို တိကျတဲ့ လုပ်ဆောင်ချက်တစ်ခုနဲ့ချိတ်ဆက်ခြင်း  

ဤ notebook သည် လက်မှတ်ထိုးထားသော အဆိုချက်နှစ်ချက်ကို စစ်ဆေးသည်- အတည်ပြုသူ key တစ်ခုက တိကျသော လုပ်ဆောင်ချက်ကို လက်မှတ်ထိုးပြီး၊ အေးဂျင့် key တစ်ခုကလည်း အဲဒီလိုလုပ်ဆောင်ချက်တူညီတဲ့ အဆိုတစ်ခုကို လက်မှတ်ထိုးထားသည်။ ၎င်းသည် provider ကို အသုံးမပြု၊ approval ကို ထုတ်ယူခြင်းမပြု၊ သို့မဟုတ် အကောင်အထည်ဖော်မှုကို မသက်သေပြသပေးပါ။ ဖန်တီးထားသော key များသည် ခွဲခြားသတ်မှတ်ထားသော အတည်ပြုသူနှင့် အေးဂျင့် key များကို အစားထိုးထားသည်။ ဥပမာတွင် WebAuthn မပြုစုဘဲ လူသားတစ်ဦးတက်ရှိမှုကို သက်သေမပြသပါ။  

နှစ်ခုစလုံး၏ ကိရိယာများသည် အခန်း၏ flat envelope ကို အသုံးပြုသည်- အမျိုးအစားတိကျသော ကလေး payload ကို Ed25519 ဖြင့် canonical JCS bytes အပေါ် လက်မှတ်ထိုးပြီး၊ လက်မှတ်ထိုးထားသော bytes များအပြင်တွင် ဖွဲ့စည်းထားသော လက်မှတ် object တစ်ခုပါရှိသည်။ approval အမျိုးအစား human.approval.v1 သည် ပညာသင်ဆန်းသစ်မှုအတွက် ဖွဲ့စည်းထားသော composition ဖြစ်ပြီး draft-farley-acta-signed-receipts မှ သတ်မှတ်ထားသော အမျိုးအစားမဟုတ်ပါ။  

ခွဲခြားထားသော pinned key ရှင်းတာချုပ်များသည် သက်ဆိုင်ရာ အာဏာပိုင်နှစ်ခုကို ကြားခံစီမံထားသည်။ တစ်ခုတည်းသော embedded key ဖြင့် ယုံကြည်ခြင်း မတည်ဆောက်နိုင်ပါ။ ဖတ်ရန်သာဖြစ်သော verifier ကလည်း လုပ်ဆောင်ချက်ချိတ်ဆက်မှု၊ ထည့်သွင်းထားသော policy version နှင့် တိကျသည့်စစ်ဆေးချိန်မှာ သက်တမ်းကုန်ဆုံးမှုကို စစ်ဆေးပါသည်။ အမှန်တကယ် အကောင်အထည်ဖော်သူသည် တည်ငြိမ်၊ atomic replay ကာကွယ်မှုကို သီးခြား အကောင်အထည်ဖော်ရမည်ဖြစ်ပြီး အောင်မြင်မှု၊ မအောင်မြင်မှု၊ မသိရသော provider ရလဒ်များကို သုံးသပ်ကြည့်ရမည်ဖြစ်သည်။  


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.b64decode(s + "=" * ((4 - len(s) % 4) % 4), altchars=b"-_", validate=True)

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## တိကျသော လုပ်ဆောင်ချက်

ခွင့်ပြုချက်ယူရာမှာ အတည်ပြုလုပ်ဆောင်ချက် အရာဝတ္ထုတစ်ခုဖြစ်ပြီး၊ "ပြန်အမ်းငွေ ခွင့်ပြုပါ" ကဲ့သို့ မရိုးရှင်းသော တံဆိပ်တစ်ခုမဟုတ်ပါ။ ၎င်း၏ ပြည့်စုံသော အရာဝတ္ထုနှင့် အကျဉ်းချုပ်သည် အတည်ပြုသူသော့ကြိုးက သဘောတူညီမှု လက်မှတ်ထိုးထားသော လုပ်ဆောင်ချက်ကို ရှာဖွေရန် စစ်ဆေးသူကို ခွင့်ပြုသည်။ ၎င်းများသည် လူတစ်ဦးသည် ကြည့်မြင်ခြင်း သို့မဟုတ် နားလည်မှုကို သတ်မှတ်ပေးခြင်း မဟုတ်ပါ။ 


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## တစ်လက်အိတ်၊ အာဏာအရင်းနှစ်ခု

လက်ခံလွှာတိုင်းမှာ ရိုးရှင်းတဲ့အမျိုးအစား payload တစ်ခုပဲနဲ့ လက်မှတ်အစိတ်အပိုင်းတစ်ခုပါ ရှိပြီး၊ ၎င်းကို လက်မှတ်ထိုးထားသော bytes မှ ထုတ်ပယ်ထားသည်။ shared verify_envelope အကူအညီက ဖုန်းခေါ်သူ၏ pinned key registry ကို အသုံးပြုသည်။

- **အတည်ပြုလက်ခံလွှာ**: အတည်ပြုသူ key သည် လုပ်ဆောင်ချက်အပြည့်အစုံ၊ ၎င်း၏ digest၊ နည်းပညာဗားရှင်းနှင့် ထုတ်ပြန်ချိန်/သက်တမ်းကုန်ဆုံးချိန် timestamp များကို လက်မှတ်ထိုးသည်။
- **အေးဂျင့်လက်ခံလွှာ**: အေးဂျင့် key သည် သောအတူတူလုပ်ဆောင်ချက် digest, တင်ပြချက်ရလဒ်နှင့် အချိန်ကာလနှင့် parent_approval_ref (အတည်ပြုချက်၏ hash) ကို လက်မှတ်ထိုးသည်။

shared action_digest သည် ပေါင်းစပ်ချက်ဖြစ်သည်။ key_id သည် pinned registry ထဲသို့ ရှာဖွေလမ်းညွှန်ချက်တစ်ခုသာ ဖြစ်သည်။ ပြန်စစ်ခြင်းသည် အရာဝတ္ထုတိုင်းကို မပျက်စီးစေ။


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the canonical JCS
    bytes directly; the signature object is NOT part of the signed bytes."""
    canonical = canonicalize(payload)
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(canonical).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    key_id = sig_obj.get("key_id")
    signature = sig_obj.get("sig")
    if not isinstance(key_id, str) or not isinstance(signature, str):
        return (False, "signature malformed (key_id and sig must be strings)")
    try:
        pub = trusted_keys.get(key_id)
    except (TypeError, ValueError):
        return (False, "invalid pinned key registry")
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized and passed directly to Ed25519.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        canonical = canonicalize(payload)
        VerifyKey(b64url_decode(pub)).verify(canonical, b64url_decode(signature))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (demo key)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## ဖတ်သည့်-သာ စာရင်းသွင်းခံယူချက်စက်ချိတ်ဆက်မှုအား အတည်ပြုခြင်း

verify_chain သည် လက်မှတ်နှစ်ခုလုံး၊ canonical လုပ်ဆောင်မှုချိတ်ဆက်မှု၊ မူဝါဒ/သော့ချက်အသစ်မြောက်မှုနှင့် သက်တမ်းကုန်ဆုံးမှုတို့ကို စစ်ဆေးပါသည်။ ၎င်းသည် စာရင်းသွင်းပြချက်အား ပေးအပ်ထားသော လုပ်ဆောင်မှုနှင့် အတည်ပြုချိန်နှင့်နှိုင်းယှဉ်သည်။ ဥပမာတွင် ISO-8601 ပုံစံတူမျိုး normalized UTC အချိန်မှတ်တမ်းများကို အသုံးပြုသည်။

ထပ်မံ၍ အတည်ပြုခြင်းသည် အတူတူခွင့်ပြုချက်များနှင့် ယုံကြည်မှုပုံစံအတိုင်း တူညီသောဖြေရှင်းချက်ကိုပေးရမည်။ ၎င်းသည် ပို့ဆောင်မှုလုပ်ငန်းမဟုတ်ဘဲ တစ်ကြိမ်တည်း လုပ်ဆောင်မှုမပေးပါ။ ထုတ်ကုန်ခံအကောင်အထည်ဖော်သူသည် ပံ့ပိုးရေးပစ်မှတ်တွင် ကိုယ်ပိုင်ခံနိုင်ရည်ရှိသော ကြိုတင်သိမ်းဆည်းမှု/စားသုံးမှု အခြေအနေရှိရပါမည်။


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, claimed_at, sk=agent_sk, key_id="agent-key-1"):
    claimed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": claimed_action,
        "action_digest": sha256_canonical(claimed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "claimed_performed",
        "claimed_at": claimed_at,
    }
    return sign_receipt(payload, sk, key_id)


def verify_chain(claimed_action, approval, agent_rcpt, now: str):
    """Read-only verification of signed claims, not execution or consumption.
    The example supplies normalized ISO-8601 UTC timestamps for comparison."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the claimed action
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the claimed action changed".
    claimed_digest = sha256_canonical(claimed_action)
    if approval.get("action_digest") != claimed_digest or approval.get("action") != claimed_action:
        return (False, "digest substitution: the approval binds a different canonical action than the claimed action")
    if agent_rcpt.get("action_digest") != claimed_digest or agent_rcpt.get("action") != claimed_action:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the claimed action")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before verification")

    return (True, "matching signed claims; execution and replay status not established")

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(verify_chain(action, approval, receipt, now="2026-07-08T15:04:07Z"))


(True, 'matching signed claims; execution and replay status not established')


## ဘိုင်တင်းက ဖမ်းဆည်းထားတာဘာတွေလဲ

အောက်ပါစစ်ဆေးမှုတွေက ကြပ်တည်းခြင်း၊ အစားထိုးခြင်း၊ လက်မှတ်ကူးခြင်း၊ ဖောင်မသင့်တဲ့ လက်ခံစာတွေ၊ သက်တမ်းဖယ်ရှားသည့် မူဝါဒ/သော့များနဲ့ သက်တမ်းကုန်ဆုံးသွားတဲ့ အတည်ပြုချက်တွေကို ငြင်းဆန်ပစ်လိုက်တယ်။ အဲဒီတွေအပြင် တရားဝင်ဆက်စပ်မှုတစ်ခုတည်းကို ထပ်ခါထပ်ခါ စစ်ဆေးနိုင်တယ်လည်း ပြသပေးတယ်။ ထပ်ခါထပ်ခါ သက်သေပြသခြင်းဟာ ပြန်လည်အသုံးပြုမှုကာကွယ်မှု မဟုတ်ပါဘူး။ အကောင်အထည်ဖော်သူကတော့ မူးယစ် dispatch များကို သီးခြားငြင်းပယ်ရမည်ဖြစ်သည်။


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the claimed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented as evidence for action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. Repeated offline verification is valid. It does not authorize a second dispatch.
print("repeat-verification ->", verify_chain(action, approval, receipt, NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (demo key)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved after approval. Verification uses the supplied current policy.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but verification came too late.
expired = human_approval(action, "alice@ops (demo key)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     claims action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (demo key)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # claiming `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the claimed action')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the claimed action')
repeat-verification -> (True, 'matching signed claims; execution and replay status not established')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the claimed action')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-rece

## ဤအရာသည် သတ်မှတ်ပေးသည့် နှင့် မသတ်မှတ်ပေးသည့်အရာများ

**သတ်မှတ်ပေးသည်။** ပေးအပ်ထားသော ပင်ထားသော key များနှင့် မူဝါဒအောက်တွင်၊ အတည်ပြုသူ key သည် ဤထိပ်တန်း canonical လုပ်ဆောင်မှုကို တိကျစွာ ချဉ်းကပ်ရေး လက်မှတ်ထိုးပြီး ဂျင် agent key သည် အတည်ပြုချက်နှင့် ကိုက်ညီသော ကြေညာချက်တစ်ခုကို လက်မှတ်ထိုးထားသည်။ သက်တမ်းကုန်လွန်မှု စစ်ဆေးခြင်းသည် ပေးအပ်ထားသော အတည်ပြုချိန်ကို အသုံးပြုသည်။ မတူညီသော ဆိုးရွားမှုအကြောင်းရင်းများသည် သက်တမ်းကုန်ပြီးပြီသော အာဏာသို့မဟုတ် ပြောင်းလဲသွားသော အဆိုပြုချက်ကို ဖော်ပြသည်။

**သတ်မှတ်ပေးမှုမရှိပါ။** လူတစ်ယောက်သည် တစုံတရာကို မြင်ကြသည် သို့မဟုတ် အတည်ပြုထားသည်၊ လက်မှတ်ထိုးသော key များသည် လုံခြုံသည်ဟု၊ အဆိုပြုချိန်ခွင့်အချိန်များသည် ယုံကြည်စိတ်ချရသည်ဟု သို့မဟုတ် ပံ့ပိုးသူတစ်ဦးသည် လုပ်ဆောင်မှုကို ဆောင်ရွက်ခဲ့သည်ဟု မသတ်မှတ်ပေးပါ။ ရလဒ်မှာ ပံ့ပိုးသူ အတည်ပြုချက်မဟုတ်ဘဲ agent လက်မှတ်ထိုးထားသော အဆိုဖြစ်သည်။

ဤ notebook သည် လုပ်ဆောင်ချက်များကို မပို့ဆောင်ပေးခြင်း သို့မဟုတ် အတည်ပြုချက်များကို မသုံးစွဲပါ။ ထုတ်လုပ်မှုပြန်လည်ဖျက်စီးမှုကာကွယ်ရန်သည် executor အနားတွင် တည်ငြိမ်ပြီး atomic အခြေအနေပြောင်းလဲမှုလိုအပ်သည်။ ပံ့ပိုးသူအနေနဲ့ အလုပ်လက်ခံပြီးနောက် ရုပ်သိမ်းမှုတစ်ခုလုံး ပျက်ဆုံးပါက ပြန်လည်ညှိနှိုင်းရန် မလုပ်မီ နိုင်ငံတကာမသေချာပတ်သက်မှုအနေဖြင့် ကျန်ရှိသင့်ပြီး ပြန်လည်ကြိုးစားခွင့် မရနိုင်ပါ။

အဆိုပါ လက်ခံလက်မှတ် နှစ်မျိုးသည် အထုပ်အပေါက် စစ်ဆေးခြင်းလမ်းကြောင်းတစ်ခုကို အမှန်ပေးထားသော စာရင်းမှတ်တမ်းအောက်၌ စီမံအုပ်ချုပ်မှုခွဲဝေထားသည်။ ၎င်းသည် လက်ခံမှု စစ်ဆေးခြင်းအား လုပ်ဆောင်မှု အာဏာအဖြစ် မပြောင်းလဲပဲ လွတ်လပ်စွာ စစ်ဆေးနိုင်ရန် ဖြစ်သည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
